# 实验7：LSTM网络实现 —— 古诗生成

本实验基于教材代码示例 7-16 至 7-31，使用 LSTM 网络实现古诗生成任务。

由于外部诗歌数据文件可能不可用，本 notebook 使用**合成的小规模古诗数据集**来演示完整的 LSTM 工作流程。

## 一、实验原理

### 1.1 RNN 的局限性

标准 RNN 在处理长序列时存在**梯度消失/爆炸**问题，无法有效学习长距离依赖。

**直觉理解**：在反向传播过程中，梯度需要沿时间步逐步往回传递，每经过一个时间步都要乘以一个权重矩阵：

- **梯度消失**：如果每次乘以一个小于1的数（比如0.5），经过100步后 $0.5^{100} \approx 10^{-31}$，梯度趋近于零——模型"记不住"远处的信息。
- **梯度爆炸**：如果每次乘以一个大于1的数（比如2.0），经过100步后 $2.0^{100} \approx 10^{30}$，梯度趋近于无穷大——训练变得不稳定。

这意味着：假设模型处理"**床**前明月光疑是地上霜举头望明月低头思故**乡**"，当模型试图学习"第1个字'床'对预测第20个字'乡'的影响"时，梯度要从第20步传回第1步，经过19次乘法，如果梯度消失了，模型就无法学到二者的关联。

### 1.2 LSTM 的门控机制

LSTM 通过三个门来控制信息流动，可以用**白板**来类比理解：

- **遗忘门（Forget Gate）** —— "擦白板"：$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$
  > 决定擦掉白板上哪些旧内容。$f_t$ 在0~1之间，0="完全擦掉"，1="完全保留"。例如读到句号时，遗忘门可能擦掉上一句的主语信息，因为新句子有新的主语。

- **输入门（Input Gate）** —— "写白板"：$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$，$\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$
  > 决定把哪些新内容写到白板上。$i_t$ 决定"要不要写"，$\tilde{C}_t$ 决定"写什么"。例如读到"明月"，输入门可能把"夜晚/思念"等语义写到白板上。

- **输出门（Output Gate）** —— "读白板"：$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$，$h_t = o_t \odot \tanh(C_t)$
  > 决定从白板上读出什么来汇报。白板上可能记了很多内容，但当前只需输出一部分。

**细胞状态**（白板本身）的更新：$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$
> 先擦掉部分旧内容，再写上新内容。由于更新是**加法**而非乘法，梯度可以顺畅流动，解决了梯度消失问题。

### 1.3 Embedding 层

将离散的字/词索引映射为稠密的向量表示，例如将"月"(索引14)映射为128维实数向量。

**为什么不用 One-Hot？**
- **One-Hot 是稀疏高维的**：词汇表有5000个字，每个字就是一个5000维向量（只有一个1）。浪费空间，且任意两个字的距离完全相同——"月"和"日"跟"月"和"马"一样远。
- **Embedding 是稠密低维的**：5000个字映射到128维空间，每个维度都是有意义的实数。
- **语义相近的字向量也相近**：训练后，"月"和"日"的向量距离比"月"和"马"更近，因为它们在古诗中出现在相似上下文中。
- **Embedding 是学习得到的**：权重在训练过程中自动调整，不需要人工设定。

### 1.4 序列到序列训练

- 输入 X：`[字1, 字2, ..., 字N-1]`（前 N-1 个字）
- 输出 Y：`[字2, 字3, ..., 字N]`（后 N-1 个字）
- 模型学习根据前文预测下一个字

**具体示例**（以"床前明月光"为例）：

| X（输入） | Y（目标） | 模型学到的规律 |
|-----------|-----------|----------------|
| 床        | 前        | 看到"床"，预测"前" |
| 前        | 明        | 看到"床前"，预测"明" |
| 明        | 月        | 看到"床前明"，预测"月" |
| 月        | 光        | 看到"床前明月"，预测"光" |

## 二、新旧写法对照（重要）

| 功能 | 旧写法 | 新写法 |
|------|--------|--------|
| 导入 LSTM | `from keras.layers import LSTM` | `from tensorflow.keras.layers import LSTM` |
| 导入 Tokenizer | `from keras.preprocessing.text import Tokenizer` | `from tensorflow.keras.preprocessing.text import Tokenizer` |
| 导入 pad_sequences | `from keras.preprocessing.sequence import pad_sequences` | `from tensorflow.keras.preprocessing.sequence import pad_sequences` |
| 导入 Embedding | `from keras.layers import Embedding` | `from tensorflow.keras.layers import Embedding` |
| 学习率参数 | `Adam(lr=0.01)` | `Adam(learning_rate=0.01)` |
| NumPy 整型 | `np.int` | `np.int64`（NumPy >= 1.20） |

## 三、代码实现

### 步骤1：导入库

首先导入所需的库。注意使用 `tensorflow.keras` 而非独立的 `keras` 包（后者在 TF 2.0+ 中已不推荐）。

**主要依赖**：
- `numpy`：数值计算，用于数组操作
- `tensorflow`：深度学习框架，提供 Keras 高级 API
- `Tokenizer`：文本分词与编码工具
- `pad_sequences`：序列补全工具
- `LSTM, Dense, Embedding` 等：神经网络层

In [5]:
import numpy as np
import tensorflow as tf

# [新写法] 适用于 TensorFlow >= 2.0
# [旧写法] from keras.preprocessing.text import Tokenizer
# [旧写法] from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# [新写法] 适用于 TensorFlow >= 2.0
# [旧写法] from keras.layers import Input, LSTM, Dense, Embedding, Activation
# [旧写法] from keras import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Activation
from tensorflow.keras import Model

# [新写法] 适用于 TensorFlow >= 2.0
# [旧写法] from keras.optimizers import Adam
from tensorflow.keras.optimizers import Adam

# [新写法] 适用于 TensorFlow >= 2.0
# [旧写法] from keras.utils import to_categorical
from tensorflow.keras.utils import to_categorical

print(f"TensorFlow 版本: {tf.__version__}")
print(f"NumPy 版本: {np.__version__}")

2026-04-07 22:55:11.485835: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-07 22:55:12.000040: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-07 22:55:13.360421: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow 版本: 2.20.0
NumPy 版本: 2.4.2


### 步骤2：构造合成古诗数据集

由于外部 `poems_clean.txt` 可能不可用，这里手动构造一批小规模古诗数据用于演示。

在实际实验中，应使用完整的古诗数据集（约 24117 首）。

**数据准备说明**：
- 实际的 `poems_clean.txt` 文件中，每行格式为 `标题:诗句内容`，需要用 `split(':')` 分割取出诗句部分。
- 每首诗被转为**单字列表**（如 `"床前明月光"` → `['床','前','明','月','光']`），因为古诗以单字为基本单位。
- 下面的代码直接将字符串转为字符列表，模拟了从文件读取后的处理结果。

In [6]:
# 合成古诗数据（五言绝句，每首20个字）
# 实际实验中应从 poems_clean.txt 读取
raw_poems = [
    "寒随穷律变春逐鸟声开初风飘带柳晚雪间花梅",
    "白日依山尽黄河入海流欲穷千里目更上一层楼",
    "床前明月光疑是地上霜举头望明月低头思故乡",
    "春眠不觉晓处处闻啼鸟夜来风雨声花落知多少",
    "红豆生南国春来发几枝愿君多采撷此物最相思",
    "松下问童子言师采药去只在此山中云深不知处",
    "空山不见人但闻人语响返景入深林复照青苔上",
    "千山鸟飞绝万径人踪灭孤舟蓑笠翁独钓寒江雪",
    "移舟泊烟渚日暮客愁新野旷天低树江清月近人",
    "独坐幽篁里弹琴复长啸深林人不知明月来相照",
    "山中相送罢日暮掩柴扉春草明年绿王孙归不归",
    "君自故乡来应知故乡事来日绮窗前寒梅着花未",
    "鹿柴空山不见人但闻人语响返景入深林复照青苔上",
    "竹里馆独坐幽篁里弹琴复长啸深林人不知明月来相照",
    "送别山中相送罢日暮掩柴扉春草明年绿王孙归不归",
    "杂诗君自故乡来应知故乡事来日绮窗前寒梅着花未",
    "相思红豆生南国春来发几枝愿君多采撷此物最相思",
    "登鹳雀楼白日依山尽黄河入海流欲穷千里目更上层楼",
    "静夜思床前明月光疑是地上霜举头望明月低头思故乡",
    "春晓春眠不觉晓处处闻啼鸟夜来风雨声花落知多少",
    "渡汉江岭外音书断经冬复历春近乡情更怯不敢问来人",
    "秋夜寄邱员外怀君属秋夜散步咏凉天空山松子落幽人应未眠",
    "问刘十九绿蚁新醅酒红泥小火炉晚来天欲雪能饮一杯无",
    "行宫寥落古行宫宫花寂寞红白头宫女在闲坐说玄宗",
    "登乐游原向晚意不适驱车登古原夕阳无限好只是近黄昏",
    "新嫁娘词三日入厨下洗手作羹汤未谙姑食性先遣小姑尝",
    "夜思床前看月光疑是地上霜抬头望山月低头思故乡",
    "江雪千山鸟飞尽万径人踪灭孤舟蓑笠翁独钓寒江雪",
    "悯农春种一粒粟秋收万颗子四海无闲田农夫犹饿死",
    "悯农锄禾日当午汗滴禾下土谁知盘中餐粒粒皆辛苦",
    "山行远上寒山石径斜白云生处有人家停车坐爱枫林晚霜叶红于二月花",
    "清明清明时节雨纷纷路上行人欲断魂借问酒家何处有牧童遥指杏花村",
    "江南春千里莺啼绿映红水村山郭酒旗风南朝四百八十寺多少楼台烟雨中",
    "泊秦淮烟笼寒水月笼沙夜泊秦淮近酒家商女不知亡国恨隔江犹唱后庭花",
    "出塞秦时明月汉时关万里长征人未还但使龙城飞将在不教胡马度阴山",
    "芙蓉楼送辛渐寒雨连江夜入吴平明送客楚山孤洛阳亲友如相问一片冰心在玉壶",
    "凉州词葡萄美酒夜光杯欲饮琵琶马上催醉卧沙场君莫笑古来征战几人回",
    "从军行青海长云暗雪山孤城遥望玉门关黄沙百战穿金甲不破楼兰终不还",
    "夏日绝句生当作人杰死亦为鬼雄至今思项羽不肯过江东",
    "乌衣巷朱雀桥边野草花乌衣巷口夕阳斜旧时王谢堂前燕飞入寻常百姓家",
]

# 将每首诗转为字符列表（模拟从文件读取后的处理）
poems = [list(poem) for poem in raw_poems]

print(f"共 {len(poems)} 首诗")
print(f"第1首诗: {''.join(poems[0])}")
print(f"第1首诗（字符列表）: {poems[0]}")

共 40 首诗
第1首诗: 寒随穷律变春逐鸟声开初风飘带柳晚雪间花梅
第1首诗（字符列表）: ['寒', '随', '穷', '律', '变', '春', '逐', '鸟', '声', '开', '初', '风', '飘', '带', '柳', '晚', '雪', '间', '花', '梅']


### 步骤3：文本编码与序列补全（Tokenization & Padding）

**Tokenizer（分词器/词表构建器）**：
- `Tokenizer()` 创建分词器，`fit_on_texts(poems)` 扫描所有诗句，按字频从高到低为每个字分配唯一整数索引。
- `word_index` 属性返回完整的"字 → 索引"字典，例如 `{'月': 3, '不': 5, '人': 7, ...}`。索引从1开始，0被保留给 padding。
- `texts_to_sequences()` 将文本转为整数序列，如 `['床','前','明']` → `[42, 58, 14]`。
- `vocab_size = len(word_index) + 1`：加1是因为索引0被保留给 padding，不在 word_index 中。

**pad_sequences（序列补全）**：
- 不同的诗长度不同，但神经网络要求输入是**固定长度**的矩阵 → 需要将所有序列补到相同长度。
- `padding='post'`：在序列**末尾**补0。例如 `[42,58,14]` → `[42,58,14,0,0,...,0]`。
- 配合 Embedding 层的 `mask_zero=True`，模型会自动忽略这些填充的0，不影响训练。

**示例**：假设 maxlen=10，诗句"床前明月光"（5个字）编码后为 `[42,58,14,3,27]`，补全后为 `[42,58,14,3,27,0,0,0,0,0]`。

In [7]:
# [旧写法] from keras.preprocessing.text import Tokenizer
# [旧写法] from keras.preprocessing.sequence import pad_sequences
# [新写法] 已在步骤1中导入

tokenizer = Tokenizer()
tokenizer.fit_on_texts(poems)
vocab_size = len(tokenizer.word_index) + 1   # +1 为保留索引0（padding/停止词）

print(f"词汇表大小: {vocab_size}")
print(f"部分字-索引映射: ", dict(list(tokenizer.word_index.items())[:10]))

# 将文本转为数字序列
poems_digit = tokenizer.texts_to_sequences(poems)
print(f"\n第1首诗的数字序列: {poems_digit[0]}")

# 补全到统一长度
# 实际实验中 maxlen=50，这里用较小的值以适配合成数据
MAXLEN = 40
poems_digit = pad_sequences(poems_digit, maxlen=MAXLEN, padding='post')
print(f"\n补全后的形状: {poems_digit.shape}")
print(f"第1首诗补全后: {poems_digit[0]}")

词汇表大小: 408
部分字-索引映射:  {'山': 1, '不': 2, '人': 3, '来': 4, '明': 5, '月': 6, '春': 7, '花': 8, '日': 9, '上': 10}

第1首诗的数字序列: [15, 204, 71, 205, 206, 7, 207, 28, 72, 208, 209, 42, 210, 211, 212, 43, 23, 213, 8, 73]

补全后的形状: (40, 40)
第1首诗补全后: [ 15 204  71 205 206   7 207  28  72 208 209  42 210 211 212  43  23 213
   8  73   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0]


### 步骤4：构造训练数据

输入 X 是前 N-1 个字，输出 Y 是后 N-1 个字（向后错一位），模型在每个时间步学习"根据当前及之前的字，预测下一个字"。

**具体示例**（以"床前明月光"为例，假设编码为 [42, 58, 14, 3, 27]）：

| 位置 | X（输入） | Y（目标） | 含义 |
|------|-----------|-----------|------|
| 1 | 42（床） | 58（前） | 看到"床"，预测"前" |
| 2 | 58（前） | 14（明） | 看到"床前"，预测"明" |
| 3 | 14（明） | 3（月） | 看到"床前明"，预测"月" |
| 4 | 3（月） | 27（光） | 看到"床前明月"，预测"光" |

**One-Hot 编码 Y**：`to_categorical(Y, num_classes=vocab_size)` 将每个整数索引转为 One-Hot 向量。例如索引3在 vocab_size=200 时变为 `[0,0,0,1,0,...,0]`（200维向量）。Y 的最终形状为 `(诗数, 时间步, vocab_size)`，对应 `categorical_crossentropy` 损失函数的要求。

In [8]:
SEQ_LEN = MAXLEN - 1  # 序列长度 = MAXLEN - 1

# X 是前 N-1 个字，Y 是后 N-1 个字
X = poems_digit[:, :-1]
Y = poems_digit[:, 1:]

print(f"X 形状: {X.shape}")
print(f"Y 形状: {Y.shape}")

print("\nX示例\tY示例")
for i in range(10):
    print(f"{X[0][i]}\t{Y[0][i]}")

# One-Hot 编码 Y
# [旧写法] from keras.utils import to_categorical
# [新写法] 已在步骤1中导入
Y = to_categorical(Y, num_classes=vocab_size)
print(f"\nY (One-Hot) 形状: {Y.shape}")

X 形状: (40, 39)
Y 形状: (40, 39)

X示例	Y示例
15	204
204	71
71	205
205	206
206	7
7	207
207	28
28	72
72	208
208	209

Y (One-Hot) 形状: (40, 39, 408)


### 步骤5：构建 LSTM 模型

模型结构：
1. **Input**: 接收长度为 SEQ_LEN 的整数序列
2. **Embedding(vocab_size, 128, mask_zero=True)**: 查找表，将每个字索引映射为128维稠密向量。`mask_zero=True` 让模型忽略 padding 的0。输出形状 `(batch, SEQ_LEN, 128)`。
3. **LSTM(64, return_sequences=True)**: 64个隐藏单元，在每个时间步都输出隐藏状态（因为是序列到序列任务）。输出形状 `(batch, SEQ_LEN, 64)`。
4. **Dense(vocab_size)**: 全连接层，将64维向量映射为 vocab_size 维的分数（logits），每个维度对应一个字的"得分"。
5. **Softmax**: 将分数转为概率分布（所有值在0~1之间，总和为1）。

**参数量估算**（假设 vocab_size=V, embedding_dim=128, hidden_size=64）：
- Embedding 层：$V \times 128$ 个参数（查找表）
- LSTM 层：$4 \times [64 \times (64 + 128) + 64] = 4 \times (12288 + 64) = 49408$ 个参数
  - 4组权重矩阵（遗忘门、输入门、候选值、输出门），每组包含输入权重 $W_{xh}$（128×64）、隐藏权重 $W_{hh}$（64×64）、偏置 $b$（64）
- Dense 层：$64 \times V + V$ 个参数

In [9]:
# [旧写法] from keras.layers import Input, LSTM, Dense, Embedding, Activation
# [旧写法] from keras import Model
# [新写法] 已在步骤1中导入

hidden_size1 = 128   # Embedding 维度
hidden_size2 = 64    # LSTM 隐藏状态维度

inp = Input(shape=(SEQ_LEN,))

# Embedding 层：将字索引映射为 dense 向量
# mask_zero=True 让模型忽略 padding 的 0
x = Embedding(vocab_size, hidden_size1, input_length=SEQ_LEN, mask_zero=True)(inp)

# LSTM 层：处理序列
# return_sequences=True 表示输出每个时间步的结果（序列到序列）
# 如果 return_sequences=False，则只输出最后一个时间步（用于分类任务）
x = LSTM(hidden_size2, return_sequences=True)(x)

# 全连接层 + Softmax：预测下一个字的概率分布
x = Dense(vocab_size)(x)
pred = Activation('softmax')(x)

model = Model(inp, pred)
print("模型构建完成！")

/home/gawainli/miniconda/envs/tf-gpu/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1775573741.217022   44608 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


模型构建完成！


### 步骤6：查看模型结构

In [10]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 39)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 39, 128)   │     52,224 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 39)        │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 39, 64)    │     49,408 │ embedding[0][0],  │
│                     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 39, 408)   │     26,520 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 39, 408)   │          0 │ dense[0][0]       │
│ (Activation)        │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 128,152 (500.59 KB)

 Trainable params: 128,152 (500.59 KB)

 Non-trainable params: 0 (0.00 B)

### 步骤7：编译与训练

**损失函数**：
- `categorical_crossentropy`：用于多分类任务，衡量模型预测的概率分布与真实 One-Hot 标签之间的差距。值越小说明预测越准。
- 如果 Y 保持整数索引格式（不做 `to_categorical`），可改用 `sparse_categorical_crossentropy`，效果相同但更节省内存。

**优化器**：
- `Adam(learning_rate=0.01)`：Adam 综合了动量和自适应学习率的优点，是深度学习中最常用的优化器。
- `learning_rate=0.01` 控制参数更新步长：过大则训练不稳定（损失震荡），过小则收敛慢。0.01 是较激进的选择，适合快速实验。

**训练参数**：
- `epochs`：整个训练集遍历的次数。合成数据量小用50轮，实际数据用10轮即可。
- `batch_size`：每次梯度更新使用的样本数。越大训练越稳定，但需要更多内存。
- `validation_split=0.2`：留出20%数据作为验证集，监控是否过拟合（训练损失下降但验证损失上升）。

注意学习率参数的新旧写法差异：
- 旧写法：`Adam(lr=0.01)`（TF 2.11+ 已废弃）
- 新写法：`Adam(learning_rate=0.01)`

In [11]:
# [旧写法] model.compile(loss='categorical_crossentropy', optimizer=Adam(lr=0.01), metrics=['accuracy'])
# ↑ 参数 lr 在 TF 2.11+ 中已废弃

# [新写法] 适用于 TensorFlow >= 2.11
model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.01),
    metrics=['accuracy']
)

print("开始训练...")
print(f"训练数据: X={X.shape}, Y={Y.shape}")
print(f"词汇表大小: {vocab_size}")
print()

# 合成数据量小，使用较小的 batch_size
# 实际实验中 batch_size=128, epochs=10
history = model.fit(
    X, Y,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

开始训练...
训练数据: X=(40, 39), Y=(40, 39, 408)
词汇表大小: 408

Epoch 1/50


2026-04-07 22:55:52.647538: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900


4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 95ms/step - accuracy: 0.0465 - loss: 5.9946 - val_accuracy: 0.0329 - val_loss: 6.0004
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.0521 - loss: 5.6450 - val_accuracy: 0.0329 - val_loss: 6.7342
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.0493 - loss: 5.4243 - val_accuracy: 0.0329 - val_loss: 7.0190
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.0620 - loss: 5.2559 - val_accuracy: 0.0329 - val_loss: 7.1374
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.0662 - loss: 5.0741 - val_accuracy: 0.0329 - val_loss: 7.0117
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.0732 - loss: 4.8664 - val_accuracy: 0.0329 - val_loss: 7.4770
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.0831 - loss: 4.6193 - val_accuracy: 0.0412 - val_loss: 7.3264
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.1366 - loss: 4.3340 - val_accuracy: 0.0412 - val_loss: 7.6124
Epoch 9/50


### 步骤8：生成古诗

给定部分文字（用 `*` 表示待生成的字），模型逐字预测填充。

例如 `'春****'` 表示以"春"开头，后面 4 个字由模型生成。

**生成算法——贪心解码（Greedy Decoding）**：
- 在每个 `*` 位置，将已生成的序列送入模型，取当前位置概率最高的字作为预测结果。
- `y[0] = 0`：将索引0（padding 符号）的概率置零，防止模型"预测"出填充符。
- `y.argmax()`：返回概率最大的字的索引（贪心选择）。

**贪心解码的局限**：每次都选最大概率的字，生成结果缺乏多样性（同一模板每次生成结果完全相同）。

**改进方法——温度采样（Temperature Sampling）**：
- 对 logits（Softmax 之前的分数）除以温度 T，再做 Softmax：$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$
- **T < 1**（如0.3）：概率分布更"尖锐"，高概率字更突出，生成更保守、更确定。
- **T = 1**：原始概率分布，不做调整。
- **T > 1**（如1.5）：概率分布更"平坦"，低概率字也有机会被选中，生成更随机、更有创意（但也可能不通顺）。
- 然后按调整后的概率分布**随机采样**（而非总选最大值）。

**其他改进**：Beam Search（束搜索）同时保留多个候选序列，最终选综合得分最优的。

In [12]:
def generate_poem(model, tokenizer, template, maxlen):
    """
    根据模板生成诗句。
    template: 字符串，已知字用原字，待生成位置用 '*'
    例如: '春****风****' 表示每句第一个字已知
    """
    poem_index = []
    poem_text = ''
    seq_len = maxlen - 1

    for i in range(len(template)):
        current_word = template[i]

        if current_word != '*':
            # 已知字：查找其索引
            if current_word in tokenizer.word_index:
                index = tokenizer.word_index[current_word]
            else:
                # 未知字，使用索引1（最常见字）作为替代
                index = 1
                current_word = tokenizer.index_word[index]
        else:
            # 待生成字：用模型预测
            x = np.expand_dims(poem_index, axis=0)
            x = pad_sequences(x, maxlen=seq_len, padding='post')
            y = model.predict(x, verbose=0)[0, i]
            y[0] = 0  # 去掉停止词（索引0）的概率
            index = y.argmax()
            current_word = tokenizer.index_word[index]

        poem_index.append(index)
        poem_text += current_word

    return poem_text


# 生成示例
# 注意：合成数据量很小，生成质量有限，仅用于演示流程
# 实际实验使用完整数据集训练后效果会好很多

template = '春****花****月****人****'
print(f"模板: {template}")
print(f"长度: {len(template)} 字")
print()

result = generate_poem(model, tokenizer, template, MAXLEN)
print("生成结果:")
# 按五言绝句格式输出
for i in range(0, len(result), 5):
    print(result[i:i+5])

模板: 春****花****月****人****
长度: 20 字

生成结果:
春夜夜夜夜
花夜夜夜夜
月夜夜夜夜
人夜夜夜夜


In [13]:
# 再试一个模板
template2 = '山****水****云****风****'
print(f"模板: {template2}")
print()

result2 = generate_poem(model, tokenizer, template2, MAXLEN)
print("生成结果:")
for i in range(0, len(result2), 5):
    print(result2[i:i+5])

模板: 山****水****云****风****

生成结果:
山夜夜夜夜
水夜夜夜夜
云夜夜夜夜
风夜夜夜夜


### 步骤9：改进版生成（Bug 修复 + 温度采样）

**原版 `generate_poem` 存在一个索引错误，导致每次都输出相同字（如"下下下下"）：**

```python
# 原版（有 Bug）
y = model.predict(x, verbose=0)[0, i]   # ← i 是模板位置
```

由于 `padding='post'`，序列实际内容在左侧（位置 0 到 len(poem_index)-1）。  
当 `i > len(poem_index)-1` 时，`[0, i]` 落在 **padding 区（全零）**，模型看到的始终是索引0，输出固定不变。

**修复：**
```python
# 修复后
pred_pos = len(poem_index) - 1          # 已生成序列的最后一个位置
y = model.predict(x, verbose=0)[0, pred_pos]  # ← 取该位置的预测
```

**额外改进——温度采样（Temperature Sampling）：**  
原版用 `argmax` 每次选概率最高的字（贪心），不同字数不同只要条件相同结果完全一样。  
温度采样按概率分布随机采样，`temperature=0.8` 时结果更有多样性且不会太乱。

In [14]:

# ============================================================
# 改进版诗歌生成
# 修复了原版的 Bug + 增加温度采样
# ============================================================

def generate_poem_v2(model, tokenizer, template, maxlen, temperature=0.8):
    """
    改进版诗歌生成。

    【Bug 修复】
    原版: y = model.predict(x)[0, i]
      → 用的是模板下标 i，但 padding='post' 时有效数据在序列左侧，
        一旦 i 超过已生成字数，就落入 padding(0) 区，每次都得到相同输出。
    修复: y = model.predict(x)[0, len(poem_index) - 1]
      → 始终取"已生成序列最后一个字"对应的输出（即对下一个字的预测）。

    【温度采样】
    原版的 argmax 贪心解码每次都选概率最大的字，导致结果单调。
    温度采样对概率分布做缩放后随机采样，temperature 越小越保守，越大越随机。
    """
    poem_index = []
    poem_text = ''
    seq_len = maxlen - 1

    for i in range(len(template)):
        current_word = template[i]

        if current_word != '*':
            if current_word in tokenizer.word_index:
                index = tokenizer.word_index[current_word]
            else:
                index = 1
                current_word = tokenizer.index_word[index]
        else:
            if len(poem_index) == 0:
                # 无上文时随机从高频字中选
                index = np.random.randint(1, min(vocab_size, 20))
                current_word = tokenizer.index_word.get(index, tokenizer.index_word[1])
                poem_index.append(index)
                poem_text += current_word
                continue

            x = np.expand_dims(poem_index, axis=0)
            x = pad_sequences(x, maxlen=seq_len, padding='post')

            # 修复：取已生成序列最后一个位置的预测输出
            pred_pos = len(poem_index) - 1
            y = model.predict(x, verbose=0)[0, pred_pos]
            y[0] = 0  # 去掉 padding 索引

            # 温度采样
            y = np.log(y + 1e-8) / temperature
            y = np.exp(y - np.max(y))  # 数值稳定的 softmax
            y = y / y.sum()
            index = np.random.choice(len(y), p=y)
            current_word = tokenizer.index_word.get(index, tokenizer.index_word[1])

        poem_index.append(index)
        poem_text += current_word

    return poem_text


template = '春****花****月****人****'
print(f"模板: {template}")
print("=" * 40)

print("\n【原版（有 Bug）】")
result_old = generate_poem(model, tokenizer, template, MAXLEN)
for i in range(0, len(result_old), 5):
    print(result_old[i:i+5])

print("\n【改进版（修复索引 + 温度采样 T=0.8，生成3次）】")
for _ in range(3):
    result_new = generate_poem_v2(model, tokenizer, template, MAXLEN, temperature=0.8)
    for i in range(0, len(result_new), 5):
        print(result_new[i:i+5])
    print()


模板: 春****花****月****人****

【原版（有 Bug）】
春夜夜夜夜
花夜夜夜夜
月夜夜夜夜
人夜夜夜夜

【改进版（修复索引 + 温度采样 T=0.8，生成3次）】
春晓春眠不
花不觉晓处
月处闻啼觉
人闻夜鸟夜

春晓春眠不
花处闻啼鸟
月来来相望
人多少不少

春眠不觉晓
花处闻啼鸟
月来风雨落
人处花有人



## 四、SimpleRNN vs LSTM 对比

| 特征 | SimpleRNN | LSTM |
|------|-----------|------|
| 结构 | 单个循环单元 | 遗忘门+输入门+输出门+细胞状态 |
| 长距离依赖 | 难以捕捉 | 有效捕捉 |
| 梯度问题 | 容易梯度消失 | 通过门控机制缓解 |
| 参数量 | 较少 | 约4倍于 SimpleRNN |
| 适用场景 | 短序列 | 长序列、复杂依赖 |

**参数量计算公式**（$n_h$ = 隐藏状态维度，$n_x$ = 输入维度）：

- **SimpleRNN**：$n_h \times (n_h + n_x) + n_h$ 个参数
  - 其中 $n_h \times n_h$ 是隐藏到隐藏的权重 $W_{hh}$，$n_h \times n_x$ 是输入到隐藏的权重 $W_{xh}$，$n_h$ 是偏置 $b$
  - 例如 $n_h=64, n_x=128$：$64 \times (64+128) + 64 = 12352$ 个参数

- **LSTM**：$4 \times [n_h \times (n_h + n_x) + n_h]$ 个参数
  - LSTM 有4组参数（遗忘门 $f$、输入门 $i$、候选值 $\tilde{C}$、输出门 $o$），每组的结构与 SimpleRNN 相同
  - 例如 $n_h=64, n_x=128$：$4 \times 12352 = 49408$ 个参数，恰好是 SimpleRNN 的 **4倍**

**为什么 LSTM 参数多4倍却更好？**
- SimpleRNN 只有一组权重，信息只能"全部保留或全部丢失"。
- LSTM 用4组权重分别控制"遗忘什么、记住什么、输出什么"，能精细地管理长期记忆。
- 更关键的是，LSTM 的细胞状态通过**加法**更新（而非 SimpleRNN 的乘法），为梯度提供了一条"高速公路"，使得梯度可以跨越很多时间步而不消失。

**GRU（门控循环单元）** 是 LSTM 的简化版本：只有2个门（更新门和重置门），参数量约为 SimpleRNN 的 **3倍**（$3 \times [n_h \times (n_h + n_x) + n_h]$），性能与 LSTM 接近但计算更快。

## 五、思考题

1. LSTM 的三个门各自起什么作用？如果去掉遗忘门会怎样？
2. Embedding 层的作用是什么？为什么不直接用 One-Hot 编码作为输入？
3. `mask_zero=True` 的作用是什么？为什么需要它？
4. `return_sequences=True` 和 `return_sequences=False` 有什么区别？
5. 如何改进诗歌生成的质量？（提示：温度采样、Beam Search、更多数据、更深网络）